In [19]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

TRAIN_PATH = "trainingStations.csv"
DEPLOY_PATH = "deployStations.csv"
TEST_PATH  = "testStations.csv"
OUT_SUBMISSION = "submission.csv"

WEEKDAY_MAP = {
    "monday": 0, "tuesday": 1, "wednesday": 2,
    "thursday": 3, "friday": 4, "saturday": 5, "sunday": 6
}

def add_time_features(df):
    df = df.copy()
    wd = df["weekday"].astype(str).str.strip().str.lower()
    df["weekday_num"] = wd.map(WEEKDAY_MAP).fillna(0).astype(int)

    df["sin_hour"] = np.sin(2*np.pi*df["hour"]/24.0)
    df["cos_hour"] = np.cos(2*np.pi*df["hour"]/24.0)

    df["sin_wd"] = np.sin(2*np.pi*df["weekday_num"]/7.0)
    df["cos_wd"] = np.cos(2*np.pi*df["weekday_num"]/7.0)

    if "weekhour" in df.columns:
        df["sin_weekhour"] = np.sin(2*np.pi*df["weekhour"]/168.0)
        df["cos_weekhour"] = np.cos(2*np.pi*df["weekhour"]/168.0)
    return df

def add_interactions(df):
    df = df.copy()
    if "temperature.C" in df.columns:
        df["temp_x_hour"] = df["temperature.C"] * df["hour"]
    if "precipitation.l.m2" in df.columns:
        df["rain_x_hour"] = df["precipitation.l.m2"] * df["hour"]
    if "windMeanSpeed.m.s" in df.columns:
        df["wind_x_hour"] = df["windMeanSpeed.m.s"] * df["hour"]
    return df

def add_station_stats(train, dfs):
    stats = train.groupby("station").agg(st_mean=("bikes","mean"), st_std=("bikes","std"))
    for df in dfs:
        df["st_mean"] = pd.to_numeric(df["station"].map(stats["st_mean"]), errors="coerce")
        df["st_std"]  = pd.to_numeric(df["station"].map(stats["st_std"]), errors="coerce")

def fill_nans(train, deploy, test):
    global_mean = float(train["bikes"].mean())
    for df in (train, deploy, test):
        if "st_std" in df.columns:
            df["st_std"] = pd.to_numeric(df["st_std"], errors="coerce").fillna(0.0)
        if "st_mean" in df.columns:
            df["st_mean"] = pd.to_numeric(df["st_mean"], errors="coerce").fillna(global_mean)

        for col in ["temperature.C", "precipitation.l.m2", "windMeanSpeed.m.s"]:
            if col in df.columns:
                med = float(pd.to_numeric(train[col], errors="coerce").median())
                df[col] = pd.to_numeric(df[col], errors="coerce").fillna(med)

def force_same_station_categories(train, deploy, test):
    all_stations = pd.concat([train["station"], deploy["station"], test["station"]]).unique()
    cat_station = pd.CategoricalDtype(categories=sorted(all_stations), ordered=False)
    train["station"] = train["station"].astype(cat_station)
    deploy["station"] = deploy["station"].astype(cat_station)
    test["station"] = test["station"].astype(cat_station)

def add_safe_physical_features(df):
    df = df.copy()

    if "numDocks" in df.columns:
        df["occ_3h"] = df["bikes_3h_ago"] / (df["numDocks"] + 1e-6)

    if "bikes_6h_ago" in df.columns:
        df["trend_3h"] = df["bikes_3h_ago"] - df["bikes_6h_ago"]
        df["delta_3h_ago"] = df["bikes_3h_ago"] - df["bikes_6h_ago"]
    else:
        df["trend_3h"] = 0.0
        df["delta_3h_ago"] = 0.0

    return df

# -------------------- LOAD --------------------
train = pd.read_csv(TRAIN_PATH)
deploy = pd.read_csv(DEPLOY_PATH)
test = pd.read_csv(TEST_PATH)

train["delta"] = train["bikes"] - train["bikes_3h_ago"]
deploy["delta"] = deploy["bikes"] - deploy["bikes_3h_ago"]

train["station"] = train["station"].astype("category")
deploy["station"] = deploy["station"].astype("category")
test["station"] = test["station"].astype("category")

# FE
train = add_safe_physical_features(add_interactions(add_time_features(train)))
deploy = add_safe_physical_features(add_interactions(add_time_features(deploy)))
test  = add_safe_physical_features(add_interactions(add_time_features(test)))

add_station_stats(train, [train, deploy, test])
fill_nans(train, deploy, test)
force_same_station_categories(train, deploy, test)

# -------------------- FEATURES --------------------
drop_cols = {"bikes","delta","weekday"}
if "Id" in train.columns:
    drop_cols.add("Id")

features = [c for c in train.columns if c not in drop_cols]
cat_features = ["station"]

X_train = train[features]
y_train = train["delta"]
X_valid = deploy[features]
y_valid = deploy["delta"]

# -------------------- MODEL --------------------
model = lgb.LGBMRegressor(
    objective="mae",
    n_estimators=6000,
    learning_rate=0.006,
    num_leaves=256,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=50,
    random_state=42,
    force_row_wise=True
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="l1",
    categorical_feature=cat_features,
    callbacks=[lgb.early_stopping(300)]
)

# -------------------- VALID PRED (delta) --------------------
pred_valid_delta = model.predict(X_valid)

# ===== OPTION A: Calibrage alpha sur deploy (shrinkage) =====
best_alpha, best_mae = None, 1e9
for alpha in np.linspace(0.85, 1.05, 21):
    pred_valid = deploy["bikes_3h_ago"].values + alpha * pred_valid_delta
    pred_valid = np.clip(pred_valid, 0, None)
    mae = mean_absolute_error(deploy["bikes"].values, pred_valid)
    if mae < best_mae:
        best_mae = mae
        best_alpha = alpha

# baseline + score
mae_baseline = mean_absolute_error(deploy["bikes"].values, deploy["bikes_3h_ago"].values)

print("======================================")
print("Baseline MAE (3h ago):", round(mae_baseline,4))
print("Best alpha           :", best_alpha)
print("Model MAE (calibrated):", round(best_mae,4))
print("Gain                 :", round(mae_baseline - best_mae,4))
print("======================================")

# -------------------- RETRAIN + SUBMISSION --------------------
full = pd.concat([train, deploy], ignore_index=True)
best_iter = int(model.best_iteration_ or 2500)

model_final = lgb.LGBMRegressor(**{**model.get_params(), "n_estimators": best_iter})
model_final.fit(full[features], full["delta"], categorical_feature=cat_features)

pred_test_delta = model_final.predict(test[features])
pred_test = test["bikes_3h_ago"].values + best_alpha * pred_test_delta
pred_test = np.clip(pred_test, 0, None)

submission = pd.DataFrame({"Id": test["Id"].values, "bikes": pred_test})
submission.to_csv(OUT_SUBMISSION, index=False)
print("Submission saved:", OUT_SUBMISSION)

[LightGBM] [Info] Total Bins 3882
[LightGBM] [Info] Number of data points in the train set: 198120, number of used features: 36
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[2033]	valid_0's l1: 3.31219
Baseline MAE (3h ago): 4.8545
Best alpha           : 1.05
Model MAE (calibrated): 3.2851
Gain                 : 1.5694
[LightGBM] [Info] Total Bins 3895
[LightGBM] [Info] Number of data points in the train set: 201096, number of used features: 36
Submission saved: submission.csv
